# nb02 — 신규 std API 가용성·최대 리드·시간 해상도 실측

기준 발표 = 2026-07-03 12z. 지점 = 제주 솔라팜(남쪽), 대조 지점 = 서산.

핵심 질문 세 가지:
1. 세 모델에서 일사·운량 등 핵심 변수가 응답하는가?
2. 발표 주기별 최대 예측 지평(hf)과 1h/3h 해상도는?
3. (검증 중 발견) 신규 std API 응답값을 그대로 믿어도 되는가?

In [1]:
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
import numpy as np
import pandas as pd
import probe_lib as pl
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

def std_val(grp, nwp, nm, tmfc, hf, x, y, data="U", level=None):
    """캐시된 신규 std 응답에서 첫 값 (캐시에 없으면 실호출 1회)."""
    b = pl.fetch_std(grp, nwp, nm, tmfc, hf, x, y, data=data, level=level)
    if not b or "ERROR" in b:
        return None
    v = [float(t) for l in b.splitlines()
         if l.strip() and not l.startswith("#") for t in l.split()]
    return v[0] if v else None

TMFC = "2026070312"   # 본 연구의 기준 발표 (2026-07-03 12z) -- 캐시 고정
print("호출 예산 상태:", pl.budget_status())

호출 예산 상태: {'real_calls': 1901, 'hard_cap': 10000, 'remaining': 8099}


In [2]:
# 응답 포맷 원문 예시 (R030 T2) -- '#' 헤더 + 값 행렬, 헤더에 요청 echo 가 있어 검증에 유용
print(pl.fetch_std("KIMR", "R030", "T2", TMFC, 24, 550, 250, help_=1))

# fname: /ARCV/RAWD/MODL/RDPS/NE57/202607/03/12/r030_v040_easia_etc.2byte.ft024.2026070312.nc, fsize: 35173504byte
# ÀÚ·áÃ³¸® ¼Ò¿ä½Ã°£ = 0.232456
# º¯¼ö¸í = T2, unit = K, level =       0, i =       1, j =       1, map = S (lon1 = 126.8, lat1 = 33.3, lon2 = 126.8, lat2 = 33.3, x_min = 550, y_min = 250, x_max = 550, y_max = 250)
# j = 1 
 2.96330e+02 
# ÀÚ·á Ç¥Ãâ½Ã°£ Æ÷ÇÔ ¼Ò¿ä½Ã°£ = 0.232485



In [3]:
# Phase 1 가용성 스윕 결과 (availability_raw.csv 로 저장돼 있음)
av = pd.read_csv("results/availability_raw.csv")
print("모델별 응답 상태 요약:")
print(av.groupby(["model", "status"]).size().unstack(fill_value=0))
print()
print("R030/L010 에서 '변수 없음(not-found)' = 운량 계열 전부:")
print(av[(av.status == "not-found")][["model", "name"]].to_string(index=False))

모델별 응답 상태 요약:
status  not-found  ok
model                
L010            5  34
NE57            0  24
R030            5  34

R030/L010 에서 '변수 없음(not-found)' = 운량 계열 전부:
model name
 R030 LCDC
 R030 MCDC
 R030 HCDC
 R030 TCDC
 R030 TCLD
 L010 LCDC
 L010 MCDC
 L010 HCDC
 L010 TCDC
 L010 TCLD


## ★발견 1 — 신규 std API 의 전구(NE57) '화면고도 변수' 디코딩 버그

전구를 신규 std API 로 읽으면 **t2m·td2m·rh2m·u10m·v10m·u80m·v80m·gust 가 전 지구 어디를 찍어도
같은 값**(위치 불변)으로 나온다. 반면 구 pt 엔드포인트는 **같은 파일**을 읽어 올바른 값을 준다.
아래 셀이 그 증거다 (서산과 제주 솔라팜, 380 km 거리).

In [4]:
rows = []
for nm in ["t2m", "rh2m", "u10m", "gust", "dswrsfc", "tcld", "tsfc", "tmax", "tmin"]:
    v1 = std_val("KIMG", "NE57", nm, TMFC, 18, 1523, 1480)   # 솔라팜
    v2 = std_val("KIMG", "NE57", nm, TMFC, 18, 1519, 1522)   # 서산
    rows.append({"변수": nm, "솔라팜": v1, "서산": v2,
                 "판정": "고장(위치 불변)" if v1 == v2 else "정상(공간 변화)"})
pd.DataFrame(rows)

,변수,솔라팜,서산,판정
0,t2m,218.66,218.66,고장(위치 불변)
1,rh2m,100.00,100.00,고장(위치 불변)
2,u10m,-0.28,-0.28,고장(위치 불변)
3,gust,4.73,4.73,고장(위치 불변)
4,dswrsfc,666.50,755.20,정상(공간 변화)
5,tcld,1.00,0.97,정상(공간 변화)
6,tsfc,298.77,304.66,정상(공간 변화)
7,tmax,299.55,302.01,정상(공간 변화)
8,tmin,298.89,300.50,정상(공간 변화)


In [5]:
# 같은 파일을 구 pt 엔드포인트로 읽으면 정상 (t2m 이 지점별로 다르고 물리적으로 타당)
for label, lat, lon in [("솔라팜", 33.3284, 126.8366), ("서산", 36.7766, 126.4939)]:
    b = pl.fetch(pl.URL_OLD_PT, {"group": "KIMG", "nwp": "NE57", "data": "U",
        "name": "t2m,u10m", "tmfc": TMFC, "hf": "18",
        "lat": f"{lat:.4f}", "lon": f"{lon:.4f}", "disp": "A", "help": "0"})
    vals = [l.split() for l in b.splitlines() if l.strip() and not l.startswith("#")]
    print(label, {v[5].split("(")[0]: float(v[4]) for v in vals})

솔라팜 {'t2m': 298.89, 'u10m': 4.56}


서산 {'t2m': 301.71, 'u10m': 2.63}


또한 구 pt 의 t2m(진짜 2m 기온)은 항상 신규 API 의 tmax·tmin(진단 구간 최대/최소) **사이**에
들어간다 — 즉 tmax/tmin 은 정상이고 t2m 만 엉뚱한 자료(성층권 추정 온도, 전 지구 상수)를 읽는다.

**결론: 전구는 구 pt 엔드포인트로 수집을 유지해야 한다** (현행 코드가 이미 그렇게 하고 있음).

In [6]:
# Phase 2 리드·해상도 스캔 결과 요약
ls = pd.concat([pd.read_csv("results/lead_scan.csv"), pd.read_csv("results/lead_scan_ext.csv")])
summary = []
for (m, c), g in ls.groupby(["model", "cycle"]):
    okh = g[g.ok == 1].hf.tolist()
    non3 = [h for h in okh if h % 3 != 0]
    summary.append({"모델": m, "발표(UTC)": f"{c:02d}z", "최대 hf": max(okh) if okh else None,
                    "1h 해상도": "전 구간" if non3 and max(non3) >= max(okh) - 1 else ("없음(3h만)" if not non3 else f"~{max(non3)}h")})
pd.DataFrame(summary).sort_values(["모델", "발표(UTC)"])

,모델,발표(UTC),최대 hf,1h 해상도
0,L010,00z,48,전 구간
1,L010,06z,48,전 구간
2,L010,12z,48,전 구간
3,L010,18z,48,전 구간
4,NE57,00z,288,없음(3h만)
5,NE57,06z,87,없음(3h만)
6,NE57,12z,288,없음(3h만)
7,NE57,18z,87,없음(3h만)
8,R030,00z,120,전 구간
9,R030,06z,72,전 구간


## 실측 스펙 (문서와 다른 부분 굵게)

| 모델 | 발표 | 최대 리드 | 해상도 |
|---|---|---|---|
| 전구 NE57 | 00/12z | 288h (D+12) | **3h 간격만** (1h 산출물 자체가 없음) |
| 전구 NE57 | 06/18z | 87h | 3h |
| 지역 R030 | 00/12z | **120h (D+5)** (문서엔 87h) | 1h 전 구간 |
| 지역 R030 | 06/18z | **72h (D+3)** | 1h 전 구간 |
| 국지 L010 | 4주기 모두 | 48h (D+2) | 1h 전 구간 |

- 전구 336/372h 는 응답 없음 → **D+13~15.5 는 영구 소실 확정** (현행 운영은 이미 D+12 로 축소돼 있어 추가 손실은 없음).
- R030 의 120h/1h 는 구 KIMR 운영 특성과 동일 — 제주 체인(D+5 1h)이 그대로 유지 가능.